# Homework 6: Decision Trees and Ensemble Learning

Regresión sobre el dataset 2026 de eficiencia de combustible
(`fuel_efficiency_mpg`) del repositorio del curso.

- Dataset: `cohorts/2026/data/car_fuel_efficiency_2026.csv` (10 000 filas, 11 columnas).
- Entorno de referencia: Python 3.12, NumPy 2.3.3, Pandas 2.3.2,
  Scikit-Learn 1.7.2 y XGBoost 3.2.0 (las versiones del release 2026).
- Preparación: imputar faltantes con `0`, split exacto del enunciado y
  `DictVectorizer(sparse=True)`.

## Imports

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

TARGET = "fuel_efficiency_mpg"

## Carga del dataset

Se usa la copia local del release 2026 que está en `teorico/`. Si no existe,
se descarga desde GitHub.

In [2]:
DATA = Path("../teorico/cohorts/2026/data/car_fuel_efficiency_2026.csv")
if not DATA.exists():
    DATA = Path("teorico/cohorts/2026/data/car_fuel_efficiency_2026.csv")
if not DATA.exists():
    DATA = Path("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main"
                "/cohorts/2026/data/car_fuel_efficiency_2026.csv")

df = pd.read_csv(DATA)
print("filas:", df.shape[0], "| columnas:", df.shape[1])
print("sha256:", hashlib.sha256(Path(DATA).read_bytes()).hexdigest() if Path(DATA).exists() else "n/a")
print("faltantes:", df.isna().sum()[df.isna().sum() > 0].to_dict())
df.head()

filas: 10000 | columnas: 11
sha256: f091fe8bb4290b0ebf0ce66fadf07c154896e047dd86ca94474a83fc57bd942d
faltantes: {'horsepower': 877, 'acceleration': 264}


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


> Nota: en un checkout de Windows los finales de línea del CSV pueden ser
> CRLF, así que el `sha256` impreso puede no coincidir con el del release
> publicado (generado en Linux). No afecta a la lectura ni a los resultados.

## Preparación

1. Rellenar los valores faltantes con `0`.
2. Hacer el split con las llamadas exactas del enunciado.
3. Convertir a matrices con `DictVectorizer(sparse=True)`.

In [3]:
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[TARGET].to_numpy()
y_val = df_val[TARGET].to_numpy()
y_test = df_test[TARGET].to_numpy()

del df_train[TARGET]
del df_val[TARGET]
del df_test[TARGET]

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient="records"))
X_val = dv.transform(df_val.to_dict(orient="records"))
X_test = dv.transform(df_test.to_dict(orient="records"))

print("train / val / test:", len(df_train), len(df_val), len(df_test))
print("features:", len(dv.feature_names_))

train / val / test: 6000 2000 2000
features: 16


## Question 1

Árbol de decisión con `max_depth=1`. ¿Qué feature original se usa para el split?

In [4]:
tree = DecisionTreeRegressor(max_depth=1, random_state=1)
tree.fit(X_train, y_train)

split_feature = dv.feature_names_[tree.tree_.feature[0]]
original_feature = split_feature.split("=", 1)[0]
print("split one-hot:", split_feature)
print("Q1 ->", original_feature)

split one-hot: model_year
Q1 -> model_year


## Question 2

Random Forest con `n_estimators=10`, `random_state=1`, `n_jobs=-1`.
¿RMSE en validación?

In [5]:
forest_10 = RandomForestRegressor(
    n_estimators=10, random_state=1, n_jobs=-1
)
forest_10.fit(X_train, y_train)
rmse_10 = mean_squared_error(y_val, forest_10.predict(X_val)) ** 0.5

print(f"RMSE (n_estimators=10) = {rmse_10:.4f}")
print(f"Q2 -> {rmse_10:.3f}")

RMSE (n_estimators=10) = 1.8371
Q2 -> 1.837


## Question 3

Probar `n_estimators` en `[10, 50, 100, 150]` con `random_state=1`.
¿Qué valor da el menor RMSE de validación (3 decimales)?

In [6]:
forest_scores = {}
for n_estimators in [10, 50, 100, 150]:
    model = RandomForestRegressor(
        n_estimators=n_estimators, random_state=1, n_jobs=-1
    )
    model.fit(X_train, y_train)
    rmse = mean_squared_error(y_val, model.predict(X_val)) ** 0.5
    forest_scores[n_estimators] = rmse

scores_table = pd.DataFrame(
    {
        "n_estimators": list(forest_scores),
        "rmse": list(forest_scores.values()),
    }
)
scores_table["rmse_3_decimals"] = scores_table["rmse"].round(3)
scores_table

,n_estimators,rmse,rmse_3_decimals
0,10,1.837054,1.837
1,50,1.771567,1.772
2,100,1.768136,1.768
3,150,1.768804,1.769


In [7]:
best_n = min(forest_scores, key=forest_scores.get)
print("Q3 ->", best_n, "| RMSE =", round(forest_scores[best_n], 3))

Q3 -> 100 | RMSE = 1.768


## Question 4

Elegir `max_depth` en `[10, 15, 20, 25]`, probando
`n_estimators` en `[10, 50, 100, 150]` y calculando el RMSE medio.
`random_state=1`.

In [8]:
depth_scores = {}
for max_depth in [10, 15, 20, 25]:
    fold_scores = []
    for n_estimators in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=1,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        rmse = mean_squared_error(y_val, model.predict(X_val)) ** 0.5
        fold_scores.append(rmse)
    depth_scores[max_depth] = float(np.mean(fold_scores))

depth_table = pd.DataFrame(
    {"max_depth": list(depth_scores), "mean_rmse": list(depth_scores.values())}
)
depth_table

,max_depth,mean_rmse
0,10,1.755405
1,15,1.783953
2,20,1.788242
3,25,1.786317


In [9]:
best_depth = min(depth_scores, key=depth_scores.get)
print("Q4 ->", best_depth, "| mean RMSE =", round(depth_scores[best_depth], 4))

Q4 -> 10 | mean RMSE = 1.7554


## Question 5

Random Forest con `n_estimators=10`, `max_depth=20`, `random_state=1`,
`n_jobs=-1`. ¿Cuál es la feature más importante entre las 4 candidatas?

In [10]:
importance_forest = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
importance_forest.fit(X_train, y_train)

importances = dict(
    zip(dv.feature_names_, importance_forest.feature_importances_)
)
candidates = [
    "vehicle_weight",
    "horsepower",
    "acceleration",
    "engine_displacement",
]
importance_table = pd.DataFrame(
    {
        "feature": candidates,
        "importance": [importances[name] for name in candidates],
    }
).sort_values("importance", ascending=False)

best_feature = max(candidates, key=lambda name: importances[name])
print("Q5 ->", best_feature)
importance_table

Q5 ->

 vehicle_weight


,feature,importance
0,vehicle_weight,0.203879
1,horsepower,0.078017
3,engine_displacement,0.061005
2,acceleration,0.055201


## Question 6

XGBoost con `eta` en `{0.3, 0.1}`, 100 rondas, `max_depth=6`,
`min_child_weight=1`, `objective="reg:squarederror"`, `seed=1`.
¿Qué `eta` da el mejor RMSE de validación?

In [11]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_val, label=y_val)
watchlist = [(dtrain, "train"), (dvalid, "valid")]

xgb_scores = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1,
    }
    booster = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False,
    )
    predictions = booster.predict(dvalid)
    xgb_scores[eta] = mean_squared_error(y_val, predictions) ** 0.5
    print(f"eta={eta}: validation RMSE = {xgb_scores[eta]:.4f}")

best_eta = min(xgb_scores, key=xgb_scores.get)
print("Q6 ->", best_eta)

eta=0.3: validation RMSE = 1.8265


eta=0.1: validation RMSE = 1.7248
Q6 -> 0.1


## Resumen de respuestas

| Pregunta | Respuesta |
| --- | --- |
| Q1. Feature del split | **`model_year`** |
| Q2. RMSE con 10 árboles | **1.837** |
| Q3. Mejor `n_estimators` | **100** |
| Q4. Mejor `max_depth` | **10** |
| Q5. Feature más importante | **`vehicle_weight`** |
| Q6. Mejor `eta` | **0.1** |